# 4.5 — Support Vector Machines (SVM)
**Type:** Classification and Regression
**Core idea:** Find the boundary with the MAXIMUM gap (margin) between classes. Only the points sitting right at the edge of that gap (support vectors) define where the boundary goes.
**When to use:** High-dimensional data (images, text, medical scans), small-to-medium datasets, when you need a clean margin of separation.

---
## Real World Problem: Cancer Tumour Classification
### Kerala Hospital — Is This Tumour Dangerous?

A patient in Thiruvananthapuram comes in with a lump. The doctor takes a biopsy and measures 30 things about the tumour cells — their size, shape, texture, smoothness.

The question: is this tumour **malignant** or **benign**?

### What do malignant and benign mean?
- **Malignant** = cancerous. Grows uncontrollably, spreads to other organs. Life threatening if untreated. Must catch immediately.
- **Benign** = non-cancerous. Grows slowly, stays in one place, doesn't spread. Usually safe.

**Analogy:** Think of a tumour like an uninvited guest in your house.
- Benign guest: sits quietly in one room, doesn't bother anyone → no emergency.
- Malignant guest: takes over rooms, breaks things, spreads chaos → remove immediately.

**Stakes:**
- Missing a malignant tumour (False Negative) → patient goes untreated → life threatening
- False alarm on benign (False Positive) → extra tests → stressful but not deadly
- Missing malignant is 100× more costly → we must LOWER the threshold

**Why SVM fits:**
- 30 features — high dimensional → SVM handles this well
- Clear margin likely exists between malignant and benign cells in 30D space
- Works well with moderate-sized medical datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     StratifiedKFold, cross_val_score)
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, recall_score,
                             precision_score, roc_curve)

np.random.seed(42)
print("Libraries loaded.")

---
## Concept 1 — Maximum Margin: The Core Idea

### Why margin matters
Logistic Regression finds ANY separating line. There are infinitely many valid lines.
SVM finds the UNIQUE line with the LARGEST possible gap from both classes.

**Kerala highway analogy:**
Think of a highway divider in Kerala splitting north-bound and south-bound traffic.
You don't want the divider hugging one lane — you want it exactly in the middle with maximum space on both sides.
If it's too close to one lane, a small swerve causes a collision. Maximum margin = maximum safety.

### Support Vectors
The points sitting right at the edge of the margin on each side = Support Vectors.
These are the most critical points — the ones closest to the boundary.

**The crucial insight:** Remove any NON-support-vector point → boundary stays identical.
Remove a SUPPORT VECTOR → boundary shifts.

So 95% of your training data is irrelevant. Only 2-3 points on each side control the boundary.

**Analogy:** The highway divider's position is controlled by the cars in the closest lane.
The cars in the middle of the road can move freely — they don't affect the divider.

---
## Concept 2 — The C Parameter (Soft Margin)

Real data has noise. Some points overlap. No perfect line exists.
SVM allows violations — but PENALISES them. The penalty is C.

### C = the strictness dial

**Large C (strict teacher):**
"I hate mistakes. Every point MUST be on the correct side."
→ Narrow margin → complex boundary → overfits training data

**Small C (relaxed teacher):**
"A few mistakes are fine. Give me a wide, relaxed boundary."
→ Wide margin → smoother boundary → might underfit

**Concrete example:**
100 patients. 5 are unusual outliers — look healthy but are diabetic.
- C=1000: Model bends the boundary to catch all 5 outliers → damages predictions on normal patients
- C=0.01: Model ignores the 5 outliers → boundary is smooth but may miss real patterns
- C=10: Model tries hard but doesn't destroy the boundary for 5 outliers → generalises well

| C | Margin | Boundary | Risk |
|---|---|---|---|
| Very large | Narrow | Complex, wiggly | Overfitting |
| Very small | Wide | Simple, smooth | Underfitting |
| Just right | Balanced | Good generalisation | ✅ |

---
## Concept 3 — The Kernel Trick

### The problem
Some data cannot be separated by ANY straight line — even with soft margin.

### The solution — go to higher dimensions
Imagine the two classes are mixed in 2D. But if you lift the data into 3D, one class sits in a valley and the other on the hills. In 3D you can separate them with a flat horizontal plane.

**The problem:** Going to higher dimensions is computationally expensive — sometimes impossible.

**The trick:** A kernel function computes the dot product in the high-dimensional space WITHOUT ever going there.

**Google Maps analogy:**
You want to know road distance between two cities. You could drive the entire route and measure. OR use Google Maps — same answer, no actual journey.
The kernel = Google Maps. Same answer, no actual transformation.

### RBF Kernel (default)
$$K(x_i, x_j) = e^{-\gamma \|x_i - x_j\|^2}$$

Plain English:
- Points CLOSE together → kernel value near 1 (very similar)
- Points FAR apart → kernel value near 0 (very different)

### Gamma — how local is the influence?

**Large gamma:**
"I only care about my immediate neighbours. Points slightly far away don't affect me."
→ Very local → complex jagged boundary → overfitting

**Small gamma:**
"I consider points far away too. Everyone has a say."
→ Smooth global boundary → might underfit

**Voting analogy:**
Large gamma = you only listen to your street when deciding how to vote. Very local.
Small gamma = you listen to the whole district. Broader influence.

### C and gamma interact
You must tune BOTH together — never one at a time.
| | Small gamma | Large gamma |
|---|---|---|
| Small C | Very smooth, relaxed | Locally relaxed |
| Large C | Globally strict | Very complex, overfits |

---
## Concept 4 — Pipeline: Why It Exists

### The wrong way (data leakage):
```python
# WRONG
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)     # scaler sees ALL data including test!

X_train, X_test = train_test_split(X_scaled, y)
# Problem: scaler already saw X_test when computing mean and std
# Your test accuracy is now fake — optimistically wrong
```

### The right way with Pipeline:
```python
# CORRECT
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])

pipeline.fit(X_train, y_train)
# Internally:
# → scaler.fit_transform(X_train)  learns mean/std from train ONLY
# → svm.fit(scaled_X_train)

pipeline.predict(X_test)
# Internally:
# → scaler.transform(X_test)  uses train's mean/std — never sees test raw data
# → svm.predict(scaled_X_test)
```

Pipeline is an assembly line: Raw data → Scaler → SVM → Prediction
Each step happens in order, automatically, with no leakage.

---
## Concept 5 — F1 vs ROC-AUC (Why You Need Both)

**F1 answers:** How good is the model AT threshold = 0.5?
It looks at who you flagged and who you missed at ONE specific cutoff.

**AUC answers:** How good are the model's probability scores across ALL thresholds?
It asks: are malignant cases consistently getting higher scores than benign cases?

### Concrete proof they measure different things:
```
Model A scores:
  Malignant patients: [0.95, 0.90, 0.85, 0.80]  ← clearly high
  Benign patients:    [0.40, 0.30, 0.20, 0.10]  ← clearly low
  F1 at 0.5 = 1.0   AUC = 1.0

Model B scores:
  Malignant patients: [0.55, 0.52, 0.51, 0.50]  ← barely above 0.5
  Benign patients:    [0.49, 0.48, 0.47, 0.10]  ← barely below 0.5
  F1 at 0.5 = 1.0   AUC = much lower
```

Both have identical F1. But Model B is fragile — lower the threshold slightly and it collapses.
AUC reveals this. F1 at 0.5 doesn't.

**Rule:**
- Use GridSearchCV with F1/Recall to tune C and gamma
- Report AUC to fairly compare models regardless of threshold
- If AUC is high but F1 is low → threshold is wrong, fix it
- If F1 is high but AUC is low → model got lucky at 0.5, scores are unreliable

---
## Step 1 — Load and Inspect the Data

In [ ]:
# Load breast cancer dataset
# 569 patients, 30 biopsy measurements, target: 0=malignant, 1=benign
data = load_breast_cancer()

X = data.data     # shape: (569, 30)
y = data.target   # 0 = malignant (cancerous), 1 = benign (safe)

df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y

print("=== Dataset Overview ===")
print(f"Patients:  {X.shape[0]}")
print(f"Features:  {X.shape[1]} (biopsy measurements: radius, texture, smoothness...)")
print(f"\nClass distribution:")
print(f"  Malignant (0): {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"  Benign    (1): {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")

print("\n=== Feature Ranges (why scaling is mandatory) ===")
print(df.drop('target', axis=1).describe().loc[['min','max']].round(2))
# Notice: mean radius is ~6-28, mean area is ~143-2501
# Without scaling, mean area would completely dominate SVM's distance calculations

In [ ]:
# EDA — which features separate malignant from benign?
top_features = ['mean radius', 'mean texture', 'mean concavity',
                'mean smoothness', 'worst radius', 'worst concavity']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flat, top_features):
    df[df['target']==0][feat].hist(ax=ax, alpha=0.6, color='coral',
                                    bins=20, label='Malignant')
    df[df['target']==1][feat].hist(ax=ax, alpha=0.6, color='steelblue',
                                    bins=20, label='Benign')
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions — Malignant vs Benign\n'
             'Good separation = SVM will find a clear margin here', fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 2 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train (455), 20% test (114)
    random_state=42,
    stratify=y          # IMPORTANT: keeps 37% malignant ratio in BOTH sets
                        # without this, test might randomly have 20% malignant
                        # making evaluation misleading
)

print(f"Train: {X_train.shape[0]} patients")
print(f"Test:  {X_test.shape[0]} patients")
print(f"\nMalignant in train: {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)")
print(f"Malignant in test:  {(y_test==0).sum()}  ({(y_test==0).mean()*100:.1f}%)")
print("Ratios match → stratify worked correctly")

---
## Step 3 — Build the Pipeline

In [ ]:
# Pipeline chains StandardScaler → SVC in the correct order
# Scaler learns from X_train only → no data leakage into X_test

pipeline = Pipeline([
    ('scaler', StandardScaler()),   # scale all 30 features to mean=0, std=1
                                    # MANDATORY for SVM — 30 features have very
                                    # different ranges (radius vs area vs texture)
    ('svm', SVC(
        kernel='rbf',               # RBF kernel — default, works for most medical data
                                    # handles non-linear boundary in 30D space
        C=1.0,                      # starting value — we'll tune this with GridSearchCV
        gamma='scale',              # auto: gamma = 1/(n_features × X.var())
                                    # safe starting point before manual tuning
        probability=True,           # enables predict_proba()
                                    # WITHOUT this: only get 0/1 predictions
                                    # WITH this: get P(malignant) for each patient
                                    # we NEED probabilities to tune threshold
        random_state=42
    ))
])

print("Pipeline built: StandardScaler → SVC(RBF)")
print("\nNote: In GridSearchCV, we reach SVM parameters using 'svm__C', 'svm__gamma'")
print("The 'svm__' prefix is because the SVM step is named 'svm' in our Pipeline.")

---
## Step 4 — First Evaluation (Before Tuning)

In [ ]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

# [:, 0] because class 0 = malignant = the dangerous outcome we care about
# predict_proba returns [P(class0), P(class1)] for each patient
# We want P(malignant) = P(class0) = column 0
y_prob = pipeline.predict_proba(X_test)[:, 0]

print("=== First Results — Before Tuning (C=1, gamma='scale') ===")
print(classification_report(y_test, y_pred,
      target_names=['Malignant (dangerous)', 'Benign (safe)']))
print(f"ROC-AUC: {roc_auc_score(y_test==0, y_prob):.4f}")
print("\nAUC tells us how well the model RANKS malignant above benign")
print("F1 tells us how good decisions are at threshold=0.5")
print("We need both — AUC to compare models, F1 for actual decisions")

In [ ]:
# Confusion matrix — understand the types of errors
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Predicted Malignant', 'Predicted Benign'],
            yticklabels=['Actual Malignant', 'Actual Benign'])
plt.title('Confusion Matrix — Cancer Detection\nBefore Tuning', fontsize=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Malignant correctly caught (TP): {tp}  ← saved lives")
print(f"Malignant missed (FN):           {fn}  ← DANGEROUS — untreated cancer")
print(f"Benign wrongly flagged (FP):     {fp}  ← extra tests, stressful but not deadly")
print(f"Benign correctly cleared (TN):   {tn}")
print(f"\nWe caught {tp}/{tp+fn} = {tp/(tp+fn)*100:.1f}% of malignant cases")
print("Goal: push this number as high as possible by tuning")

---
## Step 5 — GridSearchCV: Tune C and gamma Together

In [ ]:
# C and gamma INTERACT — must tune together, never one at a time
# 'svm__C' syntax: step_name + __ + parameter_name (because inside Pipeline)

param_grid = {
    'svm__C':     [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 0.001, 0.01, 0.1],
    'svm__kernel':['rbf', 'linear']
}

# scoring='recall' because:
# Recall = of all ACTUAL malignant cases, how many did we catch?
# Missing a malignant case = patient goes untreated = life threatening
# We optimise recall directly — not F1 — because the cost of FN >> cost of FP
# Rule: choose scoring metric based on which error is more costly in your context
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='recall',   # optimise for catching ALL malignant cases
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print(f"\nBest parameters: {grid.best_params_}")
print(f"Best CV Recall:  {grid.best_score_*100:.2f}%")
print(f"\nMeaning: with these params, we catch {grid.best_score_*100:.1f}% of malignant cases")

---
## Step 6 — Evaluate Best Model

In [ ]:
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)

# P(malignant) = probability of being class 0
# [:, 0] = first column = class 0 = malignant
y_prob_best = best_model.predict_proba(X_test)[:, 0]

print("=== Best Model Results ===")
print(classification_report(y_test, y_pred_best,
      target_names=['Malignant (dangerous)', 'Benign (safe)']))
print(f"ROC-AUC: {roc_auc_score(y_test==0, y_prob_best):.4f}")

cm_best = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm_best.ravel()
print(f"\nMalignant caught: {tp}/{tp+fn} = {tp/(tp+fn)*100:.1f}%")
print(f"Malignant missed: {fn}  ← this number must be as low as possible")

In [ ]:
# ROC Curve — visualise performance across all thresholds
fpr, tpr, thresholds_roc = roc_curve(y_test==0, y_prob_best)
auc_score = roc_auc_score(y_test==0, y_prob_best)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='coral', linewidth=2.5,
         label=f'SVM (AUC = {auc_score:.4f})')
plt.plot([0,1],[0,1], 'k--', alpha=0.4, label='Random baseline (AUC=0.5)')
plt.fill_between(fpr, tpr, alpha=0.1, color='coral')

# Mark the operating point at threshold=0.5
idx_05 = np.argmin(np.abs(thresholds_roc - 0.5))
plt.scatter(fpr[idx_05], tpr[idx_05], s=150, color='black',
            zorder=5, label=f'Threshold=0.5')

plt.xlabel('False Positive Rate\n(Benign patients wrongly flagged)')
plt.ylabel('True Positive Rate (Recall)\n(Malignant patients caught)')
plt.title('ROC Curve — Cancer Detection\n'
          'Higher AUC = better ranking of malignant above benign', fontsize=11)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"AUC = {auc_score:.4f}")
print("Interpretation: if we pick 1 random malignant and 1 random benign patient,")
print(f"the model gives the malignant patient a higher risk score {auc_score*100:.1f}% of the time.")

---
## Step 7 — Threshold Tuning (Critical for Medical Context)

Default threshold = 0.5. But missing a cancer is 100× more costly than a false alarm.
Lowering the threshold catches more malignant cases — at the cost of more false alarms.

In [ ]:
thresholds = np.arange(0.10, 0.90, 0.05)
recalls, precisions, f1s, missed_malignant, false_alarms = [], [], [], [], []

for thresh in thresholds:
    # Flag as malignant if P(malignant) >= threshold
    y_thresh = (y_prob_best >= thresh).astype(int)

    recalls.append(recall_score(y_test==0, y_thresh, zero_division=0))
    precisions.append(precision_score(y_test==0, y_thresh, zero_division=0))
    f1s.append(f1_score(y_test==0, y_thresh, zero_division=0))

    cm_t = confusion_matrix(y_test==0, y_thresh)
    missed_malignant.append(cm_t[1][0])   # FN: malignant cases missed
    false_alarms.append(cm_t[0][1])       # FP: benign wrongly flagged

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(thresholds, recalls,    color='coral',     label='Recall (catch rate)', linewidth=2)
ax1.plot(thresholds, precisions, color='steelblue', label='Precision',           linewidth=2)
ax1.plot(thresholds, f1s,        color='green',     label='F1',                  linewidth=2)
ax1.axvline(0.5,  color='gray',  linestyle='--', alpha=0.7, label='Default (0.5)')
ax1.axvline(0.35, color='black', linestyle=':',  alpha=0.8, label='Conservative (0.35)')
ax1.set_title('Metrics vs Threshold')
ax1.set_xlabel('Threshold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(thresholds, missed_malignant, color='red',    linewidth=2, label='Malignant missed (FN) ← deadly')
ax2.plot(thresholds, false_alarms,     color='orange', linewidth=2, label='Benign wrongly flagged (FP)')
ax2.axvline(0.5,  color='gray',  linestyle='--', alpha=0.7, label='Default (0.5)')
ax2.axvline(0.35, color='black', linestyle=':',  alpha=0.8, label='Conservative (0.35)')
ax2.set_title('Business Impact vs Threshold')
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Number of patients')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Threshold Tuning — Lower Threshold = Catch More Malignant Cases', fontsize=12)
plt.tight_layout()
plt.show()

idx_05  = np.argmin(np.abs(thresholds - 0.50))
idx_035 = np.argmin(np.abs(thresholds - 0.35))

print("=== Threshold Comparison ===")
print(f"\nAt threshold = 0.50 (default):")
print(f"  Malignant missed: {missed_malignant[idx_05]}  ← untreated cancers")
print(f"  False alarms:     {false_alarms[idx_05]}")
print(f"  Recall:           {recalls[idx_05]*100:.1f}%")

print(f"\nAt threshold = 0.35 (conservative for cancer):")
print(f"  Malignant missed: {missed_malignant[idx_035]}  ← fewer untreated cancers")
print(f"  False alarms:     {false_alarms[idx_035]}  ← more extra tests (acceptable)")
print(f"  Recall:           {recalls[idx_035]*100:.1f}%")
print("\nDecision: use 0.35 — missing a cancer is far worse than an extra test")

---
## Step 8 — Cross Validation

In [ ]:
cv_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== 5-Fold Cross Validation ===")
for metric in ['accuracy', 'f1', 'recall', 'roc_auc']:
    scores = cross_val_score(best_model, X_train, y_train,
                             cv=cv_final, scoring=metric)
    print(f"  {metric:12s}: {scores.mean():.4f} ± {scores.std():.4f}"
          f"  | {[round(s,3) for s in scores]}")

print("\nSmall std = consistent model across different subsets of patients")
print("Large std = model is sensitive to which patients end up in training")

---
## Step 9 — Predict for New Patients

In [ ]:
# 3 new patients walk into the Kerala hospital today
# Their biopsy measurements are recorded

# Using the dataset's feature means as realistic values
feat_means = pd.DataFrame(X, columns=data.feature_names).mean()

new_patients = pd.DataFrame([
    # Patient 1: Ravi — concerning measurements (high radius, high concavity)
    {f: feat_means[f] * (1.4 if i < 15 else 1.0)
     for i, f in enumerate(data.feature_names)},

    # Patient 2: Priya — reassuring measurements (low radius, smooth)
    {f: feat_means[f] * (0.6 if i < 15 else 1.0)
     for i, f in enumerate(data.feature_names)},

    # Patient 3: Meena — borderline measurements
    {f: feat_means[f] * (1.05 if i < 15 else 1.0)
     for i, f in enumerate(data.feature_names)},
])

# P(malignant) for each patient — column 0 = class 0 = malignant
probs = best_model.predict_proba(new_patients)[:, 0]

# Use our conservative threshold of 0.35
decisions = ['⚠️  HIGH RISK — Immediate oncology referral'
             if p >= 0.35 else '✅ LOW RISK — Routine follow-up in 6 months'
             for p in probs]

print("=== Kerala Hospital — Biopsy Results ===")
print(f"{'Patient':<10} {'P(Malignant)':>14}  {'Decision'}")
print("-" * 65)
for name, prob, decision in zip(['Ravi', 'Priya', 'Meena'], probs, decisions):
    print(f"{name:<10} {prob*100:>13.1f}%  {decision}")

print(f"\nThreshold used: 0.35")
print("Any patient above 0.35 is referred immediately.")
print("Between 0.35-0.50 = borderline → second doctor's opinion before final call.")

---
## Summary

### All decisions made in this problem and why

| Decision | What | Why |
|---|---|---|
| StandardScaler in Pipeline | Scale all 30 features | SVM uses distances — features range from 6 to 2501, unscaled = unfair |
| `probability=True` | Enable predict_proba() | Need actual probabilities to tune threshold — 0/1 not enough |
| `[:, 0]` not `[:, 1]` | Get P(malignant) | Class 0 = malignant = dangerous outcome we care about |
| `scoring='recall'` | Optimise for catching all cancers | Missing malignant is 100× worse than false alarm |
| Threshold = 0.35 | Lower than 0.5 | Aggressively catch all malignant cases |
| `stratify=y` | Keep class ratio | 37% malignant must be maintained in both train and test |
| GridSearchCV on C AND gamma | Tune together | They interact — tuning one at a time misses the best combination |

### SVM Pros and Cons

| Pros | Cons |
|---|---|
| Excellent in high dimensions (30+ features) | Slow to train when n > 50,000 |
| Works well with clear margin of separation | Very sensitive to C and gamma — must tune carefully |
| Memory efficient — stores only support vectors | No feature importance (can't say which measurement mattered most) |
| Kernel trick handles non-linear boundaries | Probabilities need `probability=True` which slows training |
| Robust to outliers (soft margin with C) | Hard to explain to a non-technical doctor why a specific decision was made |

### The one line to remember years later
SVM doesn't just find a boundary — it finds the boundary with MAXIMUM CONFIDENCE.
The margin is the model's safety zone. Wider margin = more tolerance for noise = better generalisation.
The kernel trick lets SVM draw curved boundaries without ever doing the high-dimensional transformation explicitly.

## Practice Task — Heart Disease Prediction

In [ ]:
# A cardiologist in Chennai wants to predict heart disease
# Dataset: 303 patients, 13 features

from sklearn.datasets import load_iris   # we'll use a synthetic heart-like dataset
np.random.seed(42)
n = 600

age          = np.random.randint(30, 75, n)
cholesterol  = np.random.normal(240, 50, n).clip(150, 400)
blood_press  = np.random.normal(130, 20, n).clip(90, 200)
heart_rate   = np.random.normal(75, 15, n).clip(50, 120)
st_depression= np.random.exponential(1.0, n).clip(0, 6)
num_vessels  = np.random.randint(0, 4, n)

risk = (
    (age > 55).astype(int)            * 2 +
    (cholesterol > 280).astype(int)   * 2 +
    (blood_press > 160).astype(int)   * 2 +
    (st_depression > 2).astype(int)   * 3 +
    (num_vessels > 1).astype(int)     * 3
)
prob_disease = (risk / risk.max()) * 0.85 + 0.05
heart_disease = (np.random.rand(n) < prob_disease).astype(int)

heart_df = pd.DataFrame({
    'age': age, 'cholesterol': cholesterol, 'blood_pressure': blood_press,
    'heart_rate': heart_rate, 'st_depression': st_depression,
    'num_vessels': num_vessels, 'heart_disease': heart_disease
})

print("Heart disease dataset:", heart_df.shape)
print(f"Disease rate: {heart_df['heart_disease'].mean()*100:.1f}%")
heart_df.head()

In [ ]:
# YOUR CODE HERE

# Step 1: Split into X and y
X_h = heart_df.drop(columns=['heart_disease'])
y_h = heart_df['heart_disease']

# Step 2: Train/test split (stratified, 80/20)

# Step 3: Build Pipeline — StandardScaler → SVC(probability=True)

# Step 4: Fit and evaluate with classification_report
# Which class are you more concerned about missing?
# Adjust scoring in GridSearchCV accordingly

# Step 5: GridSearchCV — tune C, gamma, kernel
# Use svm__C, svm__gamma because they're inside Pipeline

# Step 6: Plot ROC curve

# Step 7: Threshold tuning
# Missing a heart disease patient (FN) costs how much vs false alarm?
# Choose your threshold based on that reasoning

# Step 8: Predict for these 2 patients
new_patients_h = pd.DataFrame([
    {'age':65, 'cholesterol':310, 'blood_pressure':170,
     'heart_rate':95, 'st_depression':3.5, 'num_vessels':3},   # high risk
    {'age':35, 'cholesterol':190, 'blood_pressure':115,
     'heart_rate':68, 'st_depression':0.2, 'num_vessels':0},   # low risk
])
# Print P(heart disease) and decision for each